In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Mix
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from network import HierarchicalConvEmbedding, MLP, MixerBlock
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_Mix()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("reg_dense_0").output,
    outputs=model.output
)

In [7]:
l_label = [1,2,3,4,5,6,7,8,9,12]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_3/Mix-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_3/Mix-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_3/Mix-8/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_3/Mix-8/layer_cache/base
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_2: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_3: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_4: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_5: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_6: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_7: outputs (20000, 8192), labels (20000, 1)
[Saved] reg_dense_0: outputs (20000, 128), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_3/Mix-8/layer_cache/gauss/0
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000, 1)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000, 1)


In [10]:
CACHE_DIR = "./Mix-8/w_and_b_cache"

In [11]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_3/Mix-8/layer_cache/base")


==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 14557
xi>=0 count: 14073
xi>=0 count: 13317
xi>=0 count: 13943

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15435
xi>=0 count: 16163
xi>=0 count: 15822
xi>=0 count: 15959

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15475
xi>=0 count: 16701
xi>=0 count: 16121
xi>=0 count: 16641

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16092
xi>=0 count: 16962
xi>=0 count: 16695
xi>=0 count: 17198

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16283
xi>=0 count: 17181
xi>=0 count: 16754
xi>=0 count: 17419

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16490
xi>=0 count: 17217
xi>=0 count: 16986
xi>=0 count: 17577

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count:

In [12]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [13]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_3/Mix-8/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [14]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_3/Mix-8/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.64424858 0.61081975 0.4676111  0.50676822]
Layer 1
accuracy: [0.65782914 0.69685335 0.61625494 0.64810885]
Layer 2
accuracy: [0.64381183 0.73415978 0.63529602 0.69399654]
Layer 3
accuracy: [0.68329057 0.74323105 0.69720983 0.74930853]
Layer 4
accuracy: [0.69691168 0.74279185 0.7072966  0.75819693]
Layer 5
accuracy: [0.71014342 0.7458871  0.72482175 0.77112344]
Layer 6
accuracy: [0.71778978 0.76304475 0.74678665 0.77492186]
Layer 7
accuracy: [0.84232024 0.92705072 0.92196897 0.91045792]
Layer 8
accuracy: [0.98446741 0.98928445 0.97563983 0.95985153]
Layer 9
accuracy: [0.99653289 0.96564978 0.99624126 0.96673835]


In [15]:
base

array([[0.64424858, 0.61081975, 0.4676111 , 0.50676822],
       [0.65782914, 0.69685335, 0.61625494, 0.64810885],
       [0.64381183, 0.73415978, 0.63529602, 0.69399654],
       [0.68329057, 0.74323105, 0.69720983, 0.74930853],
       [0.69691168, 0.74279185, 0.7072966 , 0.75819693],
       [0.71014342, 0.7458871 , 0.72482175, 0.77112344],
       [0.71778978, 0.76304475, 0.74678665, 0.77492186],
       [0.84232024, 0.92705072, 0.92196897, 0.91045792],
       [0.98446741, 0.98928445, 0.97563983, 0.95985153],
       [0.99653289, 0.96564978, 0.99624126, 0.96673835],
       [1.        , 1.        , 1.        , 1.        ]])

In [16]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/Mix-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.69473484 0.68617397 0.57084038 0.60433063]
Layer 1
accuracy: [0.71036596 0.73718641 0.72173495 0.70547444]
Layer 2
accuracy: [0.68838046 0.75639431 0.70591577 0.72730322]
Layer 3
accuracy: [0.70837175 0.76436189 0.74023141 0.74509021]
Layer 4
accuracy: [0.72267104 0.77617894 0.739238   0.75438463]
Layer 5
accuracy: [0.72929988 0.77886515 0.75328461 0.76964616]
Layer 6
accuracy: [0.73676446 0.79427716 0.76369644 0.79187397]
Layer 7
accuracy: [0.81062724 0.87376239 0.86889179 0.84336365]
Layer 8
accuracy: [0.91123371 0.91847125 0.91969167 0.88104293]
Layer 9
accuracy: [0.94344917 0.95659948 0.93310572 0.94415405]
Layer 0
accuracy: [0.69600084 0.68141062 0.57661694 0.6025576 ]
Layer 1
accuracy: [0.70895041 0.7367942  0.71869405 0.70574544]
Layer 2
accuracy: [0.68919001 0.75458769 0.70347957 0.72430566]
Layer 3
accuracy: [0.71242656 0.76143778 0.73794816 0.74026133]
Layer 4
accuracy: [0.72185186 0.76689954 0.73593095 0.75454476]
Layer 5
accuracy: [0.72712297 0.77515376

In [17]:
gauss

array([[0.69614482, 0.68484546, 0.57339364, 0.60347473],
       [0.71096081, 0.73878971, 0.71899718, 0.70588518],
       [0.68996517, 0.75655006, 0.70501196, 0.72635001],
       [0.71293079, 0.76330123, 0.73731831, 0.7436003 ],
       [0.72366263, 0.77230085, 0.73720489, 0.75514272],
       [0.72760014, 0.7774081 , 0.74750778, 0.77148162],
       [0.7330127 , 0.79176328, 0.76190806, 0.7922874 ],
       [0.81070786, 0.8727    , 0.86660824, 0.84428094],
       [0.90978481, 0.9180951 , 0.92095894, 0.88368828],
       [0.93865446, 0.95334594, 0.93509824, 0.94357252],
       [0.9776    , 0.9726    , 0.977275  , 0.9732    ]])

In [18]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/Mix-8/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.67422087 0.63800408 0.53010025 0.54124635]
Layer 1
accuracy: [0.69839221 0.7136117  0.69841879 0.67817702]
Layer 2
accuracy: [0.68052699 0.74682179 0.69147945 0.70893887]
Layer 3
accuracy: [0.7086865  0.74970284 0.7287318  0.73905115]
Layer 4
accuracy: [0.7268008  0.75589412 0.73555956 0.75944542]
Layer 5
accuracy: [0.7328416  0.76042729 0.74614862 0.77542511]
Layer 6
accuracy: [0.74077748 0.77364197 0.76769489 0.78419578]
Layer 7
accuracy: [0.83352408 0.897148   0.89742218 0.8707425 ]
Layer 8
accuracy: [0.94747053 0.9600873  0.95270998 0.92538121]
Layer 9
accuracy: [0.96782704 0.964184   0.97576826 0.96277877]
Layer 0
accuracy: [0.67993079 0.64074498 0.5303303  0.53924209]
Layer 1
accuracy: [0.69533867 0.71929963 0.70348514 0.67713279]
Layer 2
accuracy: [0.678193   0.75212482 0.69268341 0.71648512]
Layer 3
accuracy: [0.70733744 0.75394697 0.73094623 0.74591994]
Layer 4
accuracy: [0.72162005 0.75413749 0.73791224 0.76362617]
Layer 5
accuracy: [0.73024267 0.75949745

In [19]:
salt

array([[0.67716019, 0.64017968, 0.53134202, 0.54164447],
       [0.69718552, 0.71779505, 0.69936793, 0.67727649],
       [0.67977397, 0.74967919, 0.69236914, 0.71344625],
       [0.70945624, 0.75403056, 0.73002969, 0.74315417],
       [0.72342043, 0.75754611, 0.73530239, 0.76215637],
       [0.72962599, 0.76133167, 0.7484506 , 0.77937077],
       [0.73977307, 0.77486502, 0.76742176, 0.78627802],
       [0.83141698, 0.89718782, 0.89751841, 0.87313755],
       [0.94991699, 0.9595856 , 0.95236391, 0.92369008],
       [0.97240816, 0.9650305 , 0.97456372, 0.96164315],
       [0.99455   , 0.99067499, 0.994875  , 0.99152499]])

In [20]:
SAVE_FILE='e-Mix-8.pkl'

In [21]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [22]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [24]:
mean_var = stats_minmax_from_matrix_dict(progress)

ValueError: 期望 6 个矩阵，但收到 3 个

In [ ]:
mean_var